# Performance Metrics Analysis

This notebook analyzes actual and expected performance metrics:
- Plate discipline (swing%, chase%, contact%)
- Batted ball outcomes (launch angle, exit velo, hard hit%)
- Expected stats (xBA, xwOBA, xSLG)
- Run expectancy (delta_pitcher_run_exp)
- Strikeouts, walks, and other outcomes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'
PITCHERS = {
    'Ohtani': 'shohei_ohtani.csv',
    'Sanchez': 'cristopher_sanchez.csv',
    'Misiorowski': 'jacob_misiorowski.csv'
}

dfs = {}
for name, fname in PITCHERS.items():
    df = pd.read_csv(f'{DATA_DIR}/{fname}')
    df['player_name_clean'] = df['player_name'].str.replace('\n', ', ', regex=False)
    dfs[name] = df

In [ ]:
# Plate discipline metrics
print('PLATE DISCIPLINE COMPARISON')
print('=' * 100)
disc_rows = []
for name, df in dfs.items():
    total = len(df)
    swings = df[df['description'].str.contains('swinging_strike|hit_into_play|foul', na=False, regex=True)]
    called_strikes = df[df['description'] == 'called_strike']
    whiffs = df[df['description'].str.contains('swinging_strike', na=False)]
    contact = swings[~swings.index.isin(whiffs.index)]
    in_play = df[df['description'] == 'hit_into_play']
    ooz = df[df['zone'] > 9]
    ooz_swings = ooz[ooz['description'].str.contains('swinging_strike|hit_into_play|foul', na=False, regex=True)]
    chase_rate = len(ooz_swings) / len(ooz) * 100 if len(ooz) > 0 else 0
    swing_pct = len(swings) / total * 100
    whiff_pct = len(whiffs) / len(swings) * 100 if len(swings) > 0 else 0
    contact_pct = len(contact) / len(swings) * 100 if len(swings) > 0 else 0
    called_strike_pct = len(called_strikes) / total * 100
    disc_rows.append({
        'Pitcher': name, 'Total': total,
        'Swing%': f'{swing_pct:.1f}%', 'Whiff%': f'{whiff_pct:.1f}%',
        'Contact%': f'{contact_pct:.1f}%', 'CS%': f'{called_strike_pct:.1f}%',
        'Chase%': f'{chase_rate:.1f}%'
    })
disc_df = pd.DataFrame(disc_rows)
print(disc_df.to_string(index=False))

In [ ]:
# Pitch outcomes
print('PITCH OUTCOME DISTRIBUTION')
print('=' * 100)
outcome_rows = []
for name, df in dfs.items():
    total = len(df)
    strikes = df[df['type'] == 'S']
    balls = df[df['type'] == 'B']
    in_play = df[df['type'] == 'X']
    outcome_rows.append({
        'Pitcher': name, 'Total': total,
        'Strikes': len(strikes), 'Strike%': f'{len(strikes)/total*100:.1f}%',
        'Balls': len(balls), 'Ball%': f'{len(balls)/total*100:.1f}%',
        'In Play': len(in_play), 'IP%': f'{len(in_play)/total*100:.1f}%'
    })
outcome_df = pd.DataFrame(outcome_rows)
print(outcome_df.to_string(index=False))

In [ ]:
# Expected stats (xwOBA, xBA, xSLG)
print('EXPECTED STATS COMPARISON')
print('=' * 100)
stats_rows = []
for name, df in dfs.items():
    xwoba = df['estimated_woba_using_speedangle'].dropna()
    xba = df['estimated_ba_using_speedangle'].dropna()
    xslg = df['estimated_slg_using_speedangle'].dropna()
    stats_rows.append({
        'Pitcher': name,
        'Avg xwOBA': f'{xwoba.mean():.3f}', 'Avg xBA': f'{xba.mean():.3f}',
        'Avg xSLG': f'{xslg.mean():.3f}', 'Max xwOBA': f'{xwoba.max():.3f}',
        'N_contacts': len(xwoba)
    })
stats_df = pd.DataFrame(stats_rows)
print(stats_df.to_string(index=False))
print('\nNOTE: Higher xwOBA/xBA = better for batter = WORSE for pitcher. Lower is better.')

In [ ]:
# Batted ball outcomes
print('BATTED BALL OUTCOMES')
print('=' * 100)
bb_rows = []
for name, df in dfs.items():
    in_play = df[df['description'] == 'hit_into_play']
    if len(in_play) == 0:
        continue
    launch_speed = in_play['launch_speed'].dropna()
    launch_angle = in_play['launch_angle'].dropna()
    hard_hit = (launch_speed >= 95).sum() if len(launch_speed) > 0 else 0
    if len(launch_speed) > 0 and len(launch_angle) > 0:
        barrels = ((launch_speed >= 98) & (launch_angle >= 26) & (launch_angle <= 30)).sum()
    else:
        barrels = 0
    bb_type_counts = in_play['bb_type'].value_counts()
    bb_rows.append({
        'Pitcher': name, 'BIP': len(in_play),
        'AvgEV': f'{launch_speed.mean():.1f}' if len(launch_speed) > 0 else 'N/A',
        'AvgLA': f'{launch_angle.mean():.1f}' if len(launch_angle) > 0 else 'N/A',
        'HardHit%': f'{hard_hit/len(launch_speed)*100:.1f}%' if len(launch_speed) > 0 else 'N/A',
        'Barrels': barrels,
        'GB': bb_type_counts.get('ground_ball', 0),
        'FB': bb_type_counts.get('fly_ball', 0),
        'LD': bb_type_counts.get('line_drive', 0),
    })
bb_df = pd.DataFrame(bb_rows)
print(bb_df.to_string(index=False))

In [ ]:
# Run expectancy delta
print('RUN EXPECTANCY DELTA (delta_pitcher_run_exp)')
print('=' * 100)
rexp_rows = []
for name, df in dfs.items():
    rexp = df['delta_pitcher_run_exp'].dropna()
    rexp_total = rexp.sum()
    rexp_mean = rexp.mean()
    rexp_rows.append({
        'Pitcher': name,
        'Total RE Delta': f'{rexp_total:+.3f}',
        'Avg RE per Pitch': f'{rexp_mean:+.4f}',
        'N': len(rexp),
        'Pos (bad)': (rexp > 0).sum(),
        'Neg (good)': (rexp < 0).sum(),
    })
rexp_df = pd.DataFrame(rexp_rows)
print(rexp_df.to_string(index=False))
print('\nNOTE: Negative = pitcher reduced run expectancy (GOOD). Positive = BAD.')

In [ ]:
# Strikeout & walk analysis
print('STRIKEOUT & WALK ANALYSIS')
print('=' * 100)
kbb_rows = []
for name, df in dfs.items():
    pa_df = df[df['events'].notna() & (df['events'] != '')]
    strikeouts = pa_df[pa_df['events'] == 'strikeout']
    walks = pa_df[pa_df['events'] == 'walk']
    singles = pa_df[pa_df['events'] == 'single']
    doubles = pa_df[pa_df['events'] == 'double']
    triples = pa_df[pa_df['events'] == 'triple']
    homers = pa_df[pa_df['events'] == 'home_run']
    total_pa = len(pa_df)
    hits = len(singles) + len(doubles) + len(triples) + len(homers)
    batting_avg = hits / total_pa if total_pa > 0 else 0
    obp = (hits + len(walks)) / total_pa if total_pa > 0 else 0
    slg = (len(singles) + 2*len(doubles) + 3*len(triples) + 4*len(homers)) / total_pa if total_pa > 0 else 0
    k_pct = len(strikeouts) / total_pa * 100 if total_pa > 0 else 0
    bb_pct = len(walks) / total_pa * 100 if total_pa > 0 else 0
    kbb_rows.append({
        'Pitcher': name, 'PA': total_pa,
        'K': len(strikeouts), 'K%': f'{k_pct:.1f}%',
        'BB': len(walks), 'BB%': f'{bb_pct:.1f}%',
        'K-BB%': f'{k_pct - bb_pct:.1f}%',
        'AVG': f'{batting_avg:.3f}', 'OBP': f'{obp:.3f}',
        'SLG': f'{slg:.3f}', 'OPS': f'{obp+slg:.3f}',
        'HR': len(homers)
    })
kbb_df = pd.DataFrame(kbb_rows)
print(kbb_df.to_string(index=False))

In [ ]:
# Key metrics bar chart
fig, ax = plt.subplots(figsize=(14, 6))
names = list(dfs.keys())
colors = {'Ohtani': '#1f77b4', 'Sanchez': '#ff7f0e', 'Misiorowski': '#2ca02c'}
metric_data = {}
for name, df in dfs.items():
    pa_df = df[df['events'].notna() & (df['events'] != '')]
    k_pct = len(pa_df[pa_df['events'] == 'strikeout']) / len(pa_df) * 100 if len(pa_df) > 0 else 0
    bb_pct = len(pa_df[pa_df['events'] == 'walk']) / len(pa_df) * 100 if len(pa_df) > 0 else 0
    swings = df[df['description'].str.contains('swinging_strike|hit_into_play|foul', na=False, regex=True)]
    whiffs = df[df['description'].str.contains('swinging_strike', na=False)]
    whiff_pct = len(whiffs) / len(swings) * 100 if len(swings) > 0 else 0
    xba = df['estimated_ba_using_speedangle'].dropna().mean() * 1000
    xwoba = df['estimated_woba_using_speedangle'].dropna().mean() * 1000
    in_play = df[df['description'] == 'hit_into_play']
    launch_speed = in_play['launch_speed'].dropna()
    hard_hit_pct = (launch_speed >= 95).sum() / len(launch_speed) * 100 if len(launch_speed) > 0 else 0
    metric_data[name] = [k_pct, bb_pct, whiff_pct, xba, xwoba, hard_hit_pct]
labels = ['K%', 'BB%', 'Whiff%', 'xBA*1000\n(lower=better)', 'xwOBA*1000\n(lower=better)', 'HardHit%\n(lower=better)']
x = np.arange(len(labels))
w = 0.25
for i, name in enumerate(names):
    values = metric_data[name]
    bars = ax.bar(x + i*w - w, values, w, label=name, color=colors[name], alpha=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Value')
ax.set_title('Key Performance Metrics Comparison', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('figures/key_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()